# InsightFab AI — NAFNet-SR · 200 epochs · Google Colab (NVIDIA T4 16 GB)

Full competition training run using the **existing, verified** pipeline. This notebook does
**not** change the model, split, seed, baselines, inference contract, or benchmark.

**You must edit two things before running:**
- `REPO_URL` in **Cell 3** — your GitHub repo URL (push the project first).
- `DRIVE_DATA` in **Cell 5** — the Drive folder holding `train.zip` + `Test_NoisyLR.zip`.

Optionally edit `PERSIST` in **Cell 2** (where checkpoints/logs live on Drive).

**Checkpoint strategy:** checkpoints are written to fast local disk during training and
**auto-synced to Drive every 5 minutes** (Cell 8), because NAFNet's `resume.pt` (~0.5 GB,
written every epoch) is too large to push through the Drive mount each epoch. On a
disconnect you lose at most ~5 minutes and resume from the last synced checkpoint.

Run cells **1 → 8** in order. Re-run **Cell 9** anytime to monitor. After training finishes,
run **10 → 14**. If Colab disconnects, see **"Disconnect / Resume"** below Cell 9.

## CELL 1 — GPU verification

In [ ]:
!nvidia-smi
import torch
print("\nCUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No CUDA GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")
name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print("GPU:", name)
print(f"Total VRAM: {props.total_memory/1e9:.1f} GB")
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
if "T4" not in name:
    print(f"\n⚠️  Expected a T4 but got '{name}'. If this is not the GPU you want, go to "
          "Runtime > Change runtime type > T4 GPU and re-run. Training will still run here if you continue.")
else:
    print("\n✅ T4 confirmed.")

## CELL 2 — Mount Google Drive + persistent location
Experiment records, logs, results, and synced checkpoints live here and **survive a disconnect**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# === EDIT if you want a different Drive location for outputs ===
PERSIST = '/content/drive/MyDrive/InsightFab_run'
os.environ['PERSIST'] = PERSIST
for sub in ('weights', 'experiments', 'results', 'logs'):
    os.makedirs(f'{PERSIST}/{sub}', exist_ok=True)
print("Persistent output dir on Drive:", PERSIST)
!ls -la "$PERSIST"

## CELL 3 — Clone the repository + wire outputs
`experiments/` → Drive (tiny writes). `weights/` → **local disk** (fast); a background job in Cell 8 mirrors it to Drive. **Edit `REPO_URL`.**

In [ ]:
import os
# === EDIT: your GitHub repo URL (push the InsightFab project first) ===
REPO_URL = 'https://github.com/<your-username>/insightfab-ai.git'

os.chdir('/content')
if not os.path.isdir('/content/insightfab-ai'):
    !git clone $REPO_URL insightfab-ai
os.chdir('/content/insightfab-ai')
print("Project dir:", os.getcwd())

# experiments/ -> Drive (small json + csv). Seed with the repo's committed baseline records.
!mkdir -p "$PERSIST/experiments"
!cp -n experiments/*.json experiments/index.csv "$PERSIST/experiments/" 2>/dev/null || true
!rm -rf experiments && ln -sfn "$PERSIST/experiments" experiments

# weights/ -> local ephemeral disk (fast); Cell 8 rsyncs it to Drive every 5 min.
!mkdir -p /content/ckpt && rm -rf weights && ln -sfn /content/ckpt weights
print("\nwired:")
!ls -la weights experiments | head

## CELL 4 — Install cloud dependencies
Uses `requirements-cloud.txt` (installs only what's missing on top of Colab's CUDA-matched PyTorch — does **not** reinstall torch).

In [ ]:
!pip install -q -r requirements-cloud.txt
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0))

## CELL 5 — Dataset setup
Extracts `train.zip` + `Test_NoisyLR.zip` into `data/` with the existing `scripts/setup_data.py` (excludes macOS junk; idempotent). The original dataset is not modified. **Edit `DRIVE_DATA`.**

In [ ]:
import os
# === EDIT: Drive folder that contains train.zip AND Test_NoisyLR.zip ===
DRIVE_DATA = '/content/drive/MyDrive/semicon_dataset'
os.environ['DRIVE_DATA'] = DRIVE_DATA
print("Dataset source:"); !ls -la "$DRIVE_DATA"
!python scripts/setup_data.py --src "$DRIVE_DATA"

## CELL 6 — Dataset validation (uses the committed split — no re-split)

In [ ]:
!python scripts/validate_data.py
import json
s = json.load(open('configs/split.json'))
print("\nUsing committed split:", s['counts'], "| seed", s['seed'], "| ood_cluster", s['ood_cluster'])
print("(build_split.py is intentionally NOT run — the validated split ships in the repo.)")

## CELL 7 — Configuration verification

In [ ]:
import yaml
c = yaml.safe_load(open('configs/nafnet_sr_16gb.yaml'))
t, o = c['train'], c['optim']
print("model      :", c['model']['name'], c['model']['kwargs'])
print("epochs     :", t['epochs'])
print("batch      :", t['batch'])
print("lr_patch   :", t['lr_patch'])
print("lr         :", o['lr'], "| optimizer: AdamW | betas:", o['betas'])
print("amp        :", c['amp'])
print("ema        :", c['use_ema'], "| ema_decay:", c['ema_decay'])
print("val_every  :", t['val_every'])
assert t['epochs'] == 200, "epochs must be 200"
assert 'synthetic' not in c, "synthetic augmentation must be OFF for this run"
print("\n✅ 200 epochs, no synthetic augmentation, AMP + EMA on.")

## CELL 8 — START / RESUME training (background) + Drive sync
Fixed experiment id `nafnet_sr_16gb_e200`. Restores any Drive-synced checkpoints, then resumes the **same** experiment if `resume.pt` exists, else starts at epoch 1. Training runs in the **background** (writes to fast local disk) and a second background job mirrors checkpoints to Drive **every 5 minutes**. Checkpoints are written every epoch by the trainer.

In [ ]:
import os, time
EXP = 'nafnet_sr_16gb_e200'
os.environ['EXP'] = EXP
os.environ['LOGF'] = f"{os.environ['PERSIST']}/logs/{EXP}.log"
os.environ['DRIVE_CKPT'] = f"{os.environ['PERSIST']}/weights/{EXP}"
os.makedirs(os.environ['DRIVE_CKPT'], exist_ok=True)
os.makedirs(f'weights/{EXP}', exist_ok=True)

# Restore previously-synced checkpoints from Drive into local weights/ (for resume after a reconnect).
!cp -n "$DRIVE_CKPT"/*.pt "weights/$EXP/" 2>/dev/null || true

resume = f'weights/{EXP}/resume.pt'
if not os.path.exists(f"{os.environ['PERSIST']}/logs/{EXP}.start"):
    os.system(f"date +%s > '{os.environ['PERSIST']}/logs/{EXP}.start'")

if os.path.exists(resume):
    os.environ['TRAIN'] = f"python train.py --config configs/nafnet_sr_16gb.yaml --exp-id {EXP} --resume {resume}"
    print(f"▶ RESUMING {EXP} from {resume}")
else:
    os.environ['TRAIN'] = f"python train.py --config configs/nafnet_sr_16gb.yaml --exp-id {EXP}"
    print(f"▶ STARTING fresh: {EXP} (epoch 1)")

# background sync: mirror local checkpoints -> Drive every 5 min (rsync skips unchanged files).
os.environ['SYNC'] = f'while true; do rsync -a "weights/{EXP}/" "{os.environ["DRIVE_CKPT"]}/" 2>/dev/null; sleep 300; done'

!nohup bash -c "$TRAIN" >> "$LOGF" 2>&1 &
!nohup bash -c "$SYNC" >/dev/null 2>&1 &
time.sleep(8)
print("\nLaunched training + Drive-sync (every 5 min). Recent log:")
!tail -n 8 "$LOGF"
print("\nKeep this Colab tab active to avoid an idle disconnect. Re-run CELL 9 to monitor.")

## CELL 9 — Training monitor (re-run anytime; does not interfere)

In [ ]:
import os, re, subprocess, time
EXP = os.environ.get('EXP', 'nafnet_sr_16gb_e200')
LOGF = os.environ['LOGF']

train_alive = subprocess.run("pgrep -f train.py", shell=True, capture_output=True, text=True).stdout.strip()
print("training :", ("RUNNING pid %s" % train_alive) if train_alive else "NOT running (finished/stopped)")
print("\n=== GPU ===")
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader

txt = open(LOGF).read() if os.path.exists(LOGF) else ""
ep = re.findall(r"epoch (\d+)/(\d+)", txt)
done, total = (int(ep[-1][0]), int(ep[-1][1])) if ep else (0, 200)
try:
    start = int(open(f"{os.environ['PERSIST']}/logs/{EXP}.start").read().strip())
    elapsed = time.time() - start
    rate = elapsed / done if done else 0
    eta_h = rate * (total - done) / 3600 if rate else 0
    print(f"\nprogress : epoch {done}/{total} | elapsed {elapsed/3600:.2f} h | "
          f"~{rate:.0f}s/epoch | rough ETA {eta_h:.1f} h (approx; ignores cross-session gaps)")
except Exception:
    print(f"\nprogress : epoch {done}/{total}")

print("\n=== Drive-synced checkpoints ===")
!ls -la "$DRIVE_CKPT" 2>/dev/null
print("\n=== last log lines (loss / IID / OOD) ===")
!tail -n 12 "$LOGF"

---
## ⏸️ Disconnect / Resume — what to do if Colab drops

Colab (free) disconnects on idle (~90 min) or after long sessions (~12 h). Checkpoints are
mirrored to Drive every 5 minutes, so you lose at most ~5 minutes of progress. To continue
the **same** experiment:

1. **Reconnect** a GPU runtime: Runtime → Change runtime type → **T4 GPU** → Connect.
2. Run **Cell 1** (confirm T4) and **Cell 2** (re-mount Drive).
3. Run **Cell 3** (re-clone repo — the ephemeral `/content` is wiped on disconnect).
4. Run **Cell 4** (deps) and **Cell 5** (re-extract dataset — fast, local/ephemeral).
5. Run **Cell 6** and **Cell 7** (validate + config check).
6. Run **Cell 8** — it copies the Drive-synced checkpoints back to local disk, detects
   `resume.pt`, and **resumes at the next epoch** (same optimizer/EMA/scheduler/RNG state).
   It prints `▶ RESUMING …`, not `▶ STARTING fresh`.
7. Re-run **Cell 9** to monitor.

Never edit the config or `--exp-id` between sessions — that's what keeps it one experiment.
---

## CELL 10 — Checkpoint selection (best vs epoch-200, measured)
Does **not** assume epoch 200 is best — it evaluates both `best.pt` and `last.pt` on the real IID/OOD splits.

In [ ]:
import os, torch
from src.inference.restorer import load_restorer
from src.evaluation.validate import evaluate_model
from src.data.dataset import load_split
EXP = 'nafnet_sr_16gb_e200'; dev = 'cuda'
# make sure checkpoints are present locally (e.g. after a reconnect)
!mkdir -p weights/$EXP && cp -n "$DRIVE_CKPT"/*.pt "weights/$EXP/" 2>/dev/null || true

split = load_split()
best_meta = torch.load(f'weights/{EXP}/best.pt', map_location='cpu', weights_only=False).get('meta', {})
print("best.pt meta:", best_meta)

best, _ = load_restorer(f'weights/{EXP}/best.pt', dev)
last, _ = load_restorer(f'weights/{EXP}/last.pt', dev)
b_iid = evaluate_model(best, split['val_iid'], dev, lpips_n=len(split['val_iid']))
b_ood = evaluate_model(best, split['val_ood'], dev, lpips_n=len(split['val_ood']))
l_iid = evaluate_model(last, split['val_iid'], dev, lpips_n=len(split['val_iid']))
l_ood = evaluate_model(last, split['val_ood'], dev, lpips_n=len(split['val_ood']))
r = lambda d: {k: round(v,4) for k,v in d.items() if isinstance(v,(int,float))}
print(f"\nBEST epoch {best_meta.get('epoch')}:  IID {r(b_iid)}  OOD {r(b_ood)}")
print(f"EPOCH 200 (last.pt): IID {r(l_iid)}  OOD {r(l_ood)}")
print("\n-> weights/final.pt is copied from BEST in Cell 11 (best-validation checkpoint).")

## CELL 11 — Inference verification (existing inference.py, no source edits)

In [ ]:
import numpy as np, glob, os
EXP = 'nafnet_sr_16gb_e200'
!cp weights/$EXP/best.pt weights/final.pt
!python inference.py --input_dir data/test/NoisyLR --output_dir results/test_out --checkpoint weights/final.pt
outs = sorted(glob.glob('results/test_out/*.npy'))
ins  = sorted(glob.glob('data/test/NoisyLR/*.npy'))
a = np.load(outs[0])
print("\noutputs:", len(outs), "| sample:", os.path.basename(outs[0]),
      a.shape, a.dtype, "range", round(float(a.min()),3), round(float(a.max()),3))
print("dims 256x256 :", a.shape == (256,256))
print("dtype float32:", a.dtype == np.float32)
print("filenames preserved:", {os.path.basename(p) for p in ins} == {os.path.basename(p) for p in outs},
      "| count", len(outs))

## CELL 12 — Complete end-to-end benchmark (on the T4)

In [ ]:
EXP = 'nafnet_sr_16gb_e200'
!python benchmark.py --input_dir data/test/NoisyLR --checkpoint weights/final.pt --batch 16 | tee "$PERSIST/logs/benchmark_cloud.txt"

## CELL 13 — Save all artifacts to Drive

In [ ]:
import os
EXP = 'nafnet_sr_16gb_e200'
# final mirror of checkpoints -> Drive, plus final.pt and inference outputs
!rsync -a "weights/$EXP/" "$DRIVE_CKPT/"
!cp weights/final.pt "$PERSIST/weights/final.pt"
!mkdir -p "$PERSIST/results" && cp -r results/test_out "$PERSIST/results/" 2>/dev/null || true
print("On Drive now:")
!ls -la "$DRIVE_CKPT" 2>/dev/null
!ls -la "$PERSIST/experiments"
!ls -la "$PERSIST/logs"
print("\nDownload these for the local repo:")
print("  weights: best.pt, last.pt, resume.pt   (from", os.environ['DRIVE_CKPT'] + ")")
print("  experiments/%s.json + experiments/index.csv" % EXP)
print("  logs/%s.log + logs/benchmark_cloud.txt" % EXP)

## CELL 14 — Final summary (measured values only)

In [ ]:
import json, os
EXP = 'nafnet_sr_16gb_e200'
rec = json.load(open(f'experiments/{EXP}.json'))
bic = json.load(open('experiments/baseline_bicubic.json'))
scn = json.load(open('experiments/baseline_cnn_20260811_173144.json'))
bench = open(f"{os.environ['PERSIST']}/logs/benchmark_cloud.txt").read()

print("="*66)
print("NAFNet-SR — 200-epoch run summary (all values measured from run artifacts)")
print("="*66)
print("GPU        :", rec.get('gpu'))
print("params     :", f"{rec['params']:,}")
print("train time :", f"{rec['train_sec']/3600:.2f} h ({rec['train_sec']:.0f} s)")
print("checkpoint :", rec['checkpoint'])
def line(n, iid, ood):
    print(f"{n:11s} IID  {iid['psnr']:.2f} / {iid['ssim']:.3f} / {iid['lpips']:.3f}"
          f"   OOD  {ood['psnr']:.2f} / {ood['ssim']:.3f} / {ood['lpips']:.3f}")
print("\nComparison  (PSNR / SSIM / LPIPS):")
line("Bicubic",  bic['val_iid'], bic['val_ood'])
line("SmallCNN", scn['val_iid'], scn['val_ood'])
line("NAFNet-SR",rec['val_iid'], rec['val_ood'])
print("\nBenchmark (end-to-end, on the T4):")
for l in bench.splitlines():
    if any(k in l for k in ('END-TO-END','model-only','peak GPU','device=','GPU=')):
        print("  ", l.strip())
print("\nNOTE: every number above is read from actual run artifacts — none are hand-entered.")

---
### After this run (done locally, not on Colab)
Bring the artifacts back to the local repo, then generate the visual/failure panels
(inference-only, fits the 4 GB card; the SmallCNN baseline is already local):

```bash
python scripts/make_comparison.py --nafnet weights/final.pt \
    --smallcnn weights/smallcnn_baseline.pt --n-good 4 --n-fail 4 --out results/comparisons
```